In [1]:
!pip uninstall -y transformers accelerate
!pip install transformers==4.44.2 accelerate==0.34.2 datasets==2.21.0 evaluate rouge_score sentencepiece

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: accelerate 0.34.2
Uninstalling accelerate-0.34.2:
  Successfully uninstalled accelerate-0.34.2
  Using cached transformers-4.44.2-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-0.34.2-py3-none-any.whl.metadata (19 kB)
Using cached transformers-4.44.2-py3-none-any.whl (9.5 MB)
Using cached accelerate-0.34.2-py3-none-any.whl (324 kB)


In [2]:
import transformers, accelerate
print(transformers.__version__)
print(accelerate.__version__)

4.44.2
0.34.2


# Text Summarizer Fine-Tuning on Google Colab

This notebook is Colab-ready. It fine-tunes a summarization model on the SAMSum dialogue summarization dataset and evaluates it using ROUGE.

**Recommended Colab runtime:** `Runtime > Change runtime type > GPU`


In [3]:
# Check GPU
!nvidia-smi

Mon Jun  8 19:11:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# # Install stable package versions for Colab
# !pip install -q \
#   "transformers==4.41.2" \
#   "datasets==2.20.0" \
#   "accelerate==0.31.0" \
#   "evaluate==0.4.2" \
#   "rouge_score" \
#   "sentencepiece" \
#   "sacrebleu" \
#   "py7zr"

# # After installing, Colab may ask for runtime restart in some cases.


In [4]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    pipeline,
    set_seed
)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## 1. Quick Demo Before Fine-Tuning

In [5]:
# Small demo with a pre-trained model
demo_model_name = "google/pegasus-xsum"

demo_tokenizer = AutoTokenizer.from_pretrained(demo_model_name)
demo_model = AutoModelForSeq2SeqLM.from_pretrained(demo_model_name).to(device)

article = (
    "PG&E stated it scheduled the blackouts in response to forecasts for high winds "
    "amid dry conditions. The aim is to reduce the risk of wildfires. Nearly 800 thousand "
    "customers were scheduled to be affected by the shutoffs which were expected to last "
    "through at least midday tomorrow."
)

inputs = demo_tokenizer(article, max_length=512, truncation=True, return_tensors="pt").to(device)

summary_ids = demo_model.generate(
    **inputs,
    max_length=64,
    num_beams=4,
    length_penalty=0.8
)

print(demo_tokenizer.decode(summary_ids[0], skip_special_tokens=True))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

California's largest electricity provider has turned off power to hundreds of thousands of customers.


## 2. Load SAMSum Dataset

No GitHub ZIP download is needed. This avoids the common `404 Not Found` issue in Colab.


In [6]:
# Load SAMSum safely from Hugging Face
# If the first dataset name ever fails, the fallback usually works.
try:
    dataset_samsum = load_dataset("samsum", trust_remote_code=True)
except Exception as e:
    print("load_dataset('samsum') failed. Trying fallback dataset...")
    print("Error:", e)
    dataset_samsum = load_dataset("knkarthick/samsum")

dataset_samsum


load_dataset('samsum') failed. Trying fallback dataset...
Error: Dataset 'samsum' doesn't exist on the Hub or cannot be accessed.


Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 818
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary'],
        num_rows: 819
    })
})

In [7]:
print("Splits:", dataset_samsum.keys())

for split in dataset_samsum:
    print(split, len(dataset_samsum[split]), dataset_samsum[split].column_names)

print("\nSample Dialogue:")
print(dataset_samsum["test"][0]["dialogue"])

print("\nSample Summary:")
print(dataset_samsum["test"][0]["summary"])


Splits: dict_keys(['train', 'validation', 'test'])
train 14731 ['id', 'dialogue', 'summary']
validation 818 ['id', 'dialogue', 'summary']
test 819 ['id', 'dialogue', 'summary']

Sample Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Sample Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


## 3. Load Model for Fine-Tuning

For Colab, `google/pegasus-cnn_dailymail` works on GPU but can be slow.  
This notebook uses small training subsets so it finishes faster.


In [8]:
model_checkpoint = "google/pegasus-cnn_dailymail"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint).to(device)

print("Loaded:", model_checkpoint)


tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-cnn_dailymail and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

Loaded: google/pegasus-cnn_dailymail


## 4. Tokenize Dataset

In [9]:
max_input_length = 512
max_target_length = 128

def preprocess_function(batch):
    inputs = batch["dialogue"]
    targets = batch["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset_samsum.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_samsum["train"].column_names
)

tokenized_dataset


Map:   0%|          | 0/14731 [00:00<?, ? examples/s]

Map:   0%|          | 0/818 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 14731
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 818
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 819
    })
})

## 5. Create Small Colab-Friendly Training Subsets

You can increase these values after confirming the notebook runs successfully.


In [10]:
# Use small subsets for fast Colab testing
train_size = min(1000, len(tokenized_dataset["train"]))
val_size = min(200, len(tokenized_dataset["validation"]))
test_size = min(100, len(tokenized_dataset["test"]))

small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(train_size))
small_val_dataset = tokenized_dataset["validation"].shuffle(seed=42).select(range(val_size))
small_test_dataset = dataset_samsum["test"].shuffle(seed=42).select(range(test_size))

print("Train:", len(small_train_dataset))
print("Validation:", len(small_val_dataset))
print("Test:", len(small_test_dataset))


Train: 1000
Validation: 200
Test: 100


## 6. Train Model

In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./pegasus-samsum",
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_steps=100,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    evaluation_strategy="steps",
    eval_steps=250,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,

    predict_with_generate=True,
    fp16=torch.cuda.is_available(),

    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss,Validation Loss


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}


TrainOutput(global_step=125, training_loss=2.5937027587890626, metrics={'train_runtime': 494.38, 'train_samples_per_second': 2.023, 'train_steps_per_second': 0.253, 'total_flos': 380537384214528.0, 'train_loss': 2.5937027587890626, 'epoch': 1.0})

## 7. Evaluate with ROUGE

In [12]:
rouge_metric = evaluate.load("rouge")

def generate_batch_sized_chunks(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i:i + batch_size]

def calculate_rouge(dataset, model, tokenizer, batch_size=2):
    model.eval()

    dialogue_batches = list(generate_batch_sized_chunks(dataset["dialogue"], batch_size))
    summary_batches = list(generate_batch_sized_chunks(dataset["summary"], batch_size))

    predictions = []
    references = []

    for dialogue_batch, summary_batch in tqdm(zip(dialogue_batches, summary_batches), total=len(dialogue_batches)):
        inputs = tokenizer(
            dialogue_batch,
            max_length=max_input_length,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_length=max_target_length,
                num_beams=4,
                length_penalty=0.8
            )

        decoded_preds = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        predictions.extend(decoded_preds)
        references.extend(summary_batch)

    result = rouge_metric.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    return result, predictions, references

rouge_scores, preds, refs = calculate_rouge(
    small_test_dataset,
    trainer.model,
    tokenizer,
    batch_size=2
)

pd.DataFrame([rouge_scores], index=["pegasus-samsum"])


  0%|          | 0/50 [00:00<?, ?it/s]

,rouge1,rouge2,rougeL,rougeLsum
pegasus-samsum,0.384718,0.167388,0.301624,0.301776


## 8. Save Fine-Tuned Model and Tokenizer

In [13]:
save_model_path = "/content/pegasus-samsum-model"
save_tokenizer_path = "/content/pegasus-samsum-tokenizer"

trainer.model.save_pretrained(save_model_path)
tokenizer.save_pretrained(save_tokenizer_path)

print("Model saved to:", save_model_path)
print("Tokenizer saved to:", save_tokenizer_path)


Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 128, 'min_length': 32, 'num_beams': 8, 'length_penalty': 0.8, 'forced_eos_token_id': 1}


Model saved to: /content/pegasus-samsum-model
Tokenizer saved to: /content/pegasus-samsum-tokenizer


## 9. Test the Saved Model

In [14]:
loaded_tokenizer = AutoTokenizer.from_pretrained(save_tokenizer_path)
loaded_model = AutoModelForSeq2SeqLM.from_pretrained(save_model_path).to(device)

summarizer = pipeline(
    "summarization",
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

sample_text = dataset_samsum["test"][0]["dialogue"]
reference_summary = dataset_samsum["test"][0]["summary"]

gen_kwargs = {
    "max_length": 128,
    "num_beams": 4,
    "length_penalty": 0.8
}

print("Dialogue:")
print(sample_text)

print("\nReference Summary:")
print(reference_summary)

print("\nGenerated Summary:")
print(summarizer(sample_text, **gen_kwargs)[0]["summary_text"])


Your max_length is set to 128, but your input_length is only 122. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)


Dialogue:
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

Reference Summary:
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

Generated Summary:
Hannah has Betty's number. Amanda can't find it. Larry called Betty last time they were at the park together. Hannah would rather you text him.


## 10. Optional: Download Model Folder as ZIP

In [15]:
# Optional: zip and download the saved model
!zip -r pegasus-samsum-model.zip /content/pegasus-samsum-model /content/pegasus-samsum-tokenizer

from google.colab import files
files.download("pegasus-samsum-model.zip")


  adding: content/pegasus-samsum-model/ (stored 0%)
  adding: content/pegasus-samsum-model/generation_config.json (deflated 45%)
  adding: content/pegasus-samsum-model/model.safetensors (deflated 7%)
  adding: content/pegasus-samsum-model/config.json (deflated 60%)
  adding: content/pegasus-samsum-tokenizer/ (stored 0%)
  adding: content/pegasus-samsum-tokenizer/tokenizer.json (deflated 78%)
  adding: content/pegasus-samsum-tokenizer/tokenizer_config.json (deflated 94%)
  adding: content/pegasus-samsum-tokenizer/special_tokens_map.json (deflated 82%)
  adding: content/pegasus-samsum-tokenizer/spiece.model (deflated 50%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>